In [3]:
!pip install gensim

Defaulting to user installation because normal site-packages is not writeable


In [17]:
import re
import pandas as pd
import numpy as np

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [5]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [6]:
from sklearn import set_config
set_config(display="diagram")

Dataset: https://www.kaggle.com/datasets/amar5693/fake-and-real-news-dataset-4k

In [7]:
path = r"C:\Users\Rudra\Desktop\kaggle\fake-real\fake_news_dataset_4000_rows.csv"

In [8]:
df = pd.read_csv(path)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            4000 non-null   int64 
 1   title         4000 non-null   object
 2   text          4000 non-null   object
 3   author        4000 non-null   object
 4   source        4000 non-null   object
 5   topic         4000 non-null   object
 6   publish_date  4000 non-null   object
 7   year          4000 non-null   int64 
 8   month         4000 non-null   int64 
 9   day           4000 non-null   int64 
 10  word_count    4000 non-null   int64 
 11  char_count    4000 non-null   int64 
 12  title_length  4000 non-null   int64 
 13  label         4000 non-null   int64 
dtypes: int64(8), object(6)
memory usage: 437.6+ KB


In [10]:
df.sample(3)

,id,title,text,author,source,topic,publish_date,year,month,day,word_count,char_count,title_length,label
17,18,Rumors Update 18,This article discusses Rumors Update 18. Howev...,Anonymous,ClickDaily,Rumors,2023-12-27,2023,12,27,31,214,16,0
3518,3519,Technology Update 3519,This article discusses Technology Update 3519....,Sara Ahmed,The Guardian,Technology,2024-11-02,2024,11,2,29,213,22,1
3118,3119,Technology Update 3119,This article discusses Technology Update 3119....,Ayesha Khan,Reuters,Technology,2024-07-29,2024,7,29,29,213,22,1


In [12]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", "", text)
    tokens = text.split()
    return tokens

In [14]:
df["text"]

0       This article discusses Miracle Cure Update 1. ...
1       This article discusses Conspiracy Update 2. Ho...
2       This article discusses Technology Update 3. Th...
3       This article discusses Economy Update 4. The i...
4       This article discusses Conspiracy Update 5. Ho...
                              ...                        
3995    This article discusses Alien News Update 3996....
3996    This article discusses Technology Update 3997....
3997    This article discusses Politics Update 3998. T...
3998    This article discusses Alien News Update 3999....
3999    This article discusses Celebrity Gossip Update...
Name: text, Length: 4000, dtype: object

In [15]:
df["tokens"] =  df["text"].apply(preprocess)
df["tokens"]

0       [thisarticlediscussesmiraclecureupdatehowevert...
1       [thisarticlediscussesconspiracyupdatehoweverth...
2       [thisarticlediscussestechnologyupdatetheinform...
3       [thisarticlediscusseseconomyupdatetheinformati...
4       [thisarticlediscussesconspiracyupdatehoweverth...
                              ...                        
3995    [thisarticlediscussesaliennewsupdatehoweverthe...
3996    [thisarticlediscussestechnologyupdatetheinform...
3997    [thisarticlediscussespoliticsupdatetheinformat...
3998    [thisarticlediscussesaliennewsupdatehoweverthe...
3999    [thisarticlediscussescelebritygossipupdatehowe...
Name: tokens, Length: 4000, dtype: object

In [20]:
w2v_model = Word2Vec(
    sentences = df["tokens"],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [21]:
# convert each article into vector (avg word vectors)
def get_doc_vector(tokens):
    vectors = []
    
    for word in tokens:
        if word in w2v_model.wv:
            vectors.append(w2v_model.wv[word])
    
    if len(vectors) == 0:
        return np.zeros(100)
    
    return np.mean(vectors, axis=0)

In [23]:
df["doc_vector"] = df["tokens"].apply(get_doc_vector)
df["doc_vector"] 

0       [-0.007139015, 0.0012410306, -0.0071767163, -0...
1       [-0.0087274825, 0.0021301615, -0.0008735442, -...
2       [-0.00053622725, 0.00023643136, 0.0051033497, ...
3       [0.009770293, 0.008165114, 0.0012809718, 0.005...
4       [-0.0087274825, 0.0021301615, -0.0008735442, -...
                              ...                        
3995    [-0.0051562428, -0.0066683376, -0.007776835, 0...
3996    [-0.00053622725, 0.00023643136, 0.0051033497, ...
3997    [0.00816812, -0.0044430327, 0.008985434, 0.008...
3998    [-0.0051562428, -0.0066683376, -0.007776835, 0...
3999    [-0.008242678, 0.009299355, -0.00019766092, -0...
Name: doc_vector, Length: 4000, dtype: object

In [25]:
df["doc_vector"].shape

(4000,)

In [28]:
X = np.stack(df["doc_vector"].values)
X

array([[-0.00713902,  0.00124103, -0.00717672, ...,  0.00481889,
         0.00078719,  0.00301345],
       [-0.00872748,  0.00213016, -0.00087354, ..., -0.00869202,
         0.00296152, -0.0066759 ],
       [-0.00053623,  0.00023643,  0.00510335, ..., -0.00704156,
         0.00090146,  0.00639253],
       ...,
       [ 0.00816812, -0.00444303,  0.00898543, ..., -0.00453567,
         0.00406171, -0.00427018],
       [-0.00515624, -0.00666834, -0.00777684, ...,  0.005836  ,
         0.00939121,  0.00350693],
       [-0.00824268,  0.00929935, -0.00019766, ..., -0.00744759,
        -0.00250607, -0.00554986]], dtype=float32)

In [29]:
y = df["label"]

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [32]:
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(3200, 100) (800, 100)
(3200,) (800,)


In [33]:
model = LogisticRegression(max_iter=1_000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [34]:
y_pred = model.predict(X_test)

In [35]:
print(f"Accuracy { accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

Accuracy 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       402
           1       1.00      1.00      1.00       398

    accuracy                           1.00       800
   macro avg       1.00      1.00      1.00       800
weighted avg       1.00      1.00      1.00       800

